# Exercise 10: Mixed effects

This homework assignment is designed to give you practice fitting and interpreting mixed effects models.

We will be using the **LexicalData.csv** and **Items.csv** files from the *Homework/lexDat* folder in the class GitHub repository again.

This data is a subset of the [English Lexicon Project database](https://elexicon.wustl.edu/). It provides the reaction times (in milliseconds) of many subjects as they are presented with letter strings and asked to decide, as quickly and as accurately as possible, whether the letter string is a word or not. The **Items.csv** provides characteristics of the words used, namely frequency (how common is this word?) and length (how many letters?). Unlike in the previous homework, there isn't any missing data in the **LexicalData.csv** file.

*Data courtesy of Balota, D.A., Yap, M.J., Cortese, M.J., Hutchison, K.A., Kessler, B., Loftis, B., Neely, J.H., Nelson, D.L., Simpson, G.B., & Treiman, R. (2007). The English Lexicon Project. Behavior Research Methods, 39, 445-459.*

---
## 1. Loading and formatting the data (1 point)

Load in data from the **LexicalData.csv** and **Items.csv** files. As in the previous homeworks, remove the commas from the reaction times and convert them from strings to numbers. Use `left_join` to add word characteristics `Length` and `Log_Freq_Hal` from **Items** to **LexicalData**.

*Note: the `Freq_HAL` variable in **Items.csv** has a similar formatting issue, using string values with commas. We're not going to worry about fixing this since we're only using `Log_Freq_HAL`, which is the natural log transformation of `Freq_HAL`, in this homework.*

In [8]:
library("dplyr")

wd <- "/Users/kaitlyndalbon/Desktop/Carnegie_Mellon/Courses/85732_Data_Science/DataSciencePsychNeuro/Exercise datasets/lexDat/"
df_lexDat <- read.csv(paste0(wd, "LexicalData.csv"))
df_items <- read.csv(paste0(wd, "Items.csv"))

df_items <- df_items %>% select(c('Word', 'Length', 'Log_Freq_HAL'))

df <- left_join(df_lexDat, df_items, by = join_by(D_Word == Word))
df <- df %>% mutate(D_RT = as.numeric(gsub(",", "", D_RT)))
str(df)

'data.frame':	62610 obs. of  9 variables:
 $ Sub_ID      : int  157 67 120 21 236 236 236 236 236 236 ...
 $ Trial       : int  1 1 1 1 1 2 5 6 8 10 ...
 $ Type        : int  1 1 1 1 1 1 1 1 1 1 ...
 $ D_RT        : num  710 1094 587 984 577 ...
 $ D_Word      : chr  "browse" "refrigerant" "gaining" "cheerless" ...
 $ Outlier     : chr  "false" "false" "false" "false" ...
 $ D_Zscore    : num  -0.437 0.825 -0.645 0.025 -0.763 ...
 $ Length      : int  6 11 7 9 8 8 10 6 11 9 ...
 $ Log_Freq_HAL: num  8.86 4.64 8.3 2.64 1.39 ...


---
## 2. Model fitting (4 points)

First, fit a linear model with `Log_Freq_HAL` and `Length` as predictors, and `D_RT` as the output. Include an interaction term. Use `summary()` to look at the model output.

In [2]:
model_1 <- lm(D_RT ~ Log_Freq_HAL + Length + Log_Freq_HAL*Length, data = df)
summary(model_1)


Call:
lm(formula = D_RT ~ Log_Freq_HAL + Length + Log_Freq_HAL * Length, 
    data = df)

Residuals:
     Min       1Q   Median       3Q      Max 
-1118.01  -205.23   -86.95    90.77  3147.07 

Coefficients:
                    Estimate Std. Error t value Pr(>|t|)    
(Intercept)         610.1903    14.6678  41.601  < 2e-16 ***
Log_Freq_HAL         -6.0239     1.9678  -3.061  0.00221 ** 
Length               47.7531     1.6368  29.175  < 2e-16 ***
Log_Freq_HAL:Length  -2.9421     0.2348 -12.528  < 2e-16 ***
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

Residual standard error: 359.1 on 62606 degrees of freedom
Multiple R-squared:  0.09473,	Adjusted R-squared:  0.09469 
F-statistic:  2184 on 3 and 62606 DF,  p-value: < 2.2e-16


Now, install `lme4` using `install.packages()` and then load the library.

In [4]:
install.packages("lme4")
library("lme4") 


The downloaded binary packages are in
	/var/folders/5_/v0q2657s7b354r1303qrlrxw0000gp/T//RtmpVd9VUC/downloaded_packages


Now fit a mixed effects model that includes the same predictors as the linear model above, as well as random intercepts for `Sub_ID` (i.e., cases where subject ID shifts the RT mean). Use `summary()` to look at the model output.

In [5]:
model_2 <- lmer(D_RT ~ Log_Freq_HAL + Length + Log_Freq_HAL*Length + (1|Sub_ID), data=df)
summary(model_2)

Linear mixed model fit by REML ['lmerMod']
Formula: D_RT ~ Log_Freq_HAL + Length + Log_Freq_HAL * Length + (1 | Sub_ID)
   Data: df

REML criterion at convergence: 888235.6

Scaled residuals: 
    Min      1Q  Median      3Q     Max 
-4.5058 -0.5472 -0.1568  0.3103 10.7381 

Random effects:
 Groups   Name        Variance Std.Dev.
 Sub_ID   (Intercept) 46333    215.3   
 Residual             82978    288.1   
Number of obs: 62610, groups:  Sub_ID, 299

Fixed effects:
                    Estimate Std. Error t value
(Intercept)         616.8445    17.1522  35.963
Log_Freq_HAL         -7.4374     1.5830  -4.698
Length               47.7477     1.3162  36.277
Log_Freq_HAL:Length  -2.8778     0.1888 -15.239

Correlation of Fixed Effects:
            (Intr) Lg_F_HAL Length
Log_Frq_HAL -0.645                
Length      -0.656  0.917         
Lg_Fr_HAL:L  0.582 -0.942   -0.923

---
## 3. Model assessment (4 points)

Compare the three t-values for the fixed effects and the mixed effects models. How do they differ, and why?

> T-values in the mixed effects model are larger because the model accounts for subject-level random effects. By explicitly modeling within-subject variability, the mixed effects model reduces unexplained residual error and produces more precise estimates of the fixed effects. On the other hand, the fixed effects model ignores the dependence among observations within subjects, which inflates the residual variance and leads to smaller t-values. In other words, I think the mixed effects models separate within-subject and between-subject sources of variance, leading to a more efficient estimation of the fixed effects. 

Use the Aikeke Information Criterion (AIC) to compare these two models. Which one is better?

In [6]:
model_fit <- AIC(model_1, model_2)
model_fit
diff(model_fit$AIC)

,df,AIC
,<dbl>,<dbl>
model_1,5,914436.4
model_2,6,888247.6


[1] -26188.82

> Model 2 is better. 

---
##  4. Reflection (1 point)

What other random effects could be controlled for in this data set?

> Trial; participants might be more tired as the experiment progresses, resulting in increased variance in response. 

**DUE:** 11:59pm EST, March 5, 2026

**IMPORTANT** Did you collaborate with anyone on this assignment? If so, list their names here.
> I did not collaborate with any of my classmates on this assignment.

**GenAI Utilization** Did you utilize any generative AI tools on this assignment? If so, please list the item and the paste respective prompt you used.
> I did not utilize any generative AI tools on this assignment.